# TP1 - Sistema de Reconocimiento Facial: Entrenamiento
----

Alumnos: 
- Gianfranco Frattini
- Matias Prado

Este notebook documenta paso a paso la construcción y entrenamiento de nuestro sistema de reconocimiento facial, cumpliendo con los requerimientos del Trabajo Práctico N°1.

Se estructuran las 4 etapas del pipeline, la recolección de datos (incluyendo LFW), las estrategias de oversampling/data augmentation, y finalmente el entrenamiento y validación del modelo.

In [1]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import os, json, random, datetime
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
import timm
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sklearn.datasets import fetch_lfw_people
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc
from sklearn.preprocessing import label_binarize
from collections import Counter
from insightface.app import FaceAnalysis
from insightface.utils import face_align
from PIL import Image

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Usando dispositivo: {DEVICE}")
if DEVICE == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Crear carpetas de salida
os.makedirs("Graphics", exist_ok=True)
os.makedirs("train_metrics", exist_ok=True)
os.makedirs("models", exist_ok=True)

Usando dispositivo: cuda
  GPU: NVIDIA GeForce RTX 3080
  VRAM: 10.7 GB


## El Pipeline de 4 Etapas

Tal como requiere el TP, el sistema se divide en cuatro etapas claramente definidas. A continuación, definimos las funciones que luego se utilizarán en el servicio (`face_service.py`).

### 1. Detección de Rostros
Utilizamos el módulo de detección de InsightFace (`buffalo_s`) para extraer las cajas delimitadoras (*bounding boxes*) y los 5 puntos clave faciales (*keypoints*).

In [2]:
# Inicializar el analizador (solo módulo de detección)
analyzer = FaceAnalysis(name="buffalo_s", allowed_modules=["detection"])
analyzer.prepare(ctx_id=0 if DEVICE == "cuda" else -1, det_size=(640, 640))

def detect_faces(image: np.ndarray):
    """
    Detecta rostros en una imagen BGR.
    Retorna una lista de tuplas: (bbox, keypoints)
    - bbox: (x1, y1, x2, y2) coordenadas absolutas.
    - keypoints: array (5, 2) con las coordenadas de los ojos, nariz y boca.
    """
    faces = analyzer.get(image)
    result = []
    for face in faces:
        bbox = tuple(map(int, face.bbox))
        result.append((bbox, face.kps))
    return result

/usr/local/lib/python3.12/dist-packages/onnxruntime/capi/onnxruntime_inference_collection.py:123: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(


Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
model ignore: /root/.insightface/models/buffalo_s/1k3d68.onnx landmark_3d_68
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
model ignore: /root/.insightface/models/buffalo_s/2d106det.onnx landmark_2d_106
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_s/det_500m.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
model ignore: /root/.insightface/models/buffalo_s/genderage.onnx genderage
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
model ignore: /root/.insightface/models/buffalo_s/w600k_mbf.onnx recognition
set det-size: (640, 640)


### 2. Alineación Facial (Normalización Geométrica)
La alineación recorta el rostro y aplica una transformación afín para que los ojos y la boca siempre queden en la misma posición relativa, sin importar la rotación de la cabeza original. Esto simplifica el trabajo del modelo extractor.

In [3]:
def align_face(image: np.ndarray, box: tuple, kps: np.ndarray, face_size: int = 112) -> np.ndarray:
    """
    Recorta y alinea el rostro usando los keypoints.
    Retorna una imagen BGR de tamaño (face_size, face_size).
    """
    if kps is not None:
        crop = face_align.norm_crop(image, landmark=kps, image_size=face_size)
    else:
        # Fallback por si no hay keypoints
        x1, y1, x2, y2 = map(int, box)
        crop = image[max(0, y1):max(0, y2), max(0, x1):max(0, x2)]
        if crop.size > 0:
            crop = cv2.resize(crop, (face_size, face_size))
        else:
            crop = np.zeros((face_size, face_size, 3), dtype=np.uint8)
    return crop

### 3. Extracción de Embeddings (Modelo Backbone)
Utilizamos una red neuronal convolucional (ResNet-50 en este caso) preentrenada, a la que le adaptamos la última capa para emitir un vector (embedding) de dimensión fija.

In [4]:
class FaceEmbeddingModel(nn.Module):
    def __init__(self, num_classes, embedding_size=512, dropout=0.3):
        super().__init__()
        self.backbone = timm.create_model("resnet50", pretrained=True)
        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity()
        self.embedding_head = nn.Linear(in_features, embedding_size)
        self.dropout = nn.Dropout(p=dropout)
        self.classifier = nn.Linear(embedding_size, num_classes)

    def forward(self, x, return_embedding=False):
        features = self.backbone(x)
        embedding = self.embedding_head(features)
        if return_embedding:
            return nn.functional.normalize(embedding, p=2, dim=1)
        x = nn.functional.relu(embedding)
        x = self.dropout(x)
        return self.classifier(x)

def extract_embedding_from_face(model, face_rgb: np.ndarray, transform) -> np.ndarray:
    """Pasa una cara alineada (RGB) por el modelo para obtener su embedding."""
    tensor = transform(Image.fromarray(face_rgb)).unsqueeze(0).to(DEVICE)
    model.eval()
    with torch.no_grad():
        if isinstance(model, nn.Sequential):
            embedding = model(tensor)
            embedding = nn.functional.normalize(embedding, p=2, dim=1)
        else:
            embedding = model(tensor, return_embedding=True)
    return embedding.cpu().numpy()[0]

### 4. Identificación / Verificación
Compara dos embeddings utilizando la Similitud Coseno.

In [5]:
def similarity(query_emb: np.ndarray, ref_emb: np.ndarray) -> float:
    """Calcula la similitud coseno entre dos vectores."""
    denom = np.linalg.norm(query_emb) * np.linalg.norm(ref_emb)
    if denom == 0:
        return 0.0
    return float(np.dot(query_emb, ref_emb) / denom)

## Preparación del Dataset: Custom + Labeled Faces in the Wild (LFW)

Como nuestro dataset es pequeño y las imágenes locales son "capturas de pantalla", necesitamos aumentar la cantidad de imágenes.
1. Añadimos el dataset **LFW** (Presidentes y personalidades públicas).
2. Procesamos nuestro dataset local. Excluimos completamente a "Valentino" para usarlo como prueba de "desconocido".
3. Aplicamos **Oversampling** (Duplicación con Data Augmentation) a las clases minoritarias locales (ej. Ambrogi) para que el modelo no genere sesgos hacia las clases mayoritarias de LFW.

In [6]:
# 1. Descargar LFW
print("Descargando/Cargando LFW...")
lfw = fetch_lfw_people(min_faces_per_person=70, color=True, resize=1.0)
X_lfw_raw = lfw.images
y_lfw_raw = lfw.target
lfw_names = list(lfw.target_names)

print(f"LFW: {X_lfw_raw.shape[0]} imágenes, {len(lfw_names)} personas.")

X_lfw_bgr = []
for img in X_lfw_raw:
    img_uint8 = (img * 255).astype(np.uint8)
    img_bgr = cv2.cvtColor(img_uint8, cv2.COLOR_RGB2BGR)
    X_lfw_bgr.append(img_bgr)

Descargando/Cargando LFW...
LFW: 1288 imágenes, 7 personas.


In [7]:
# 2. Cargar Dataset Local
DATASET_DIR = "Dataset"
local_classes = []
X_local_bgr = []
y_local_raw = []
val_valentino_images = []

for cls_name in sorted(os.listdir(DATASET_DIR)):
    cls_path = os.path.join(DATASET_DIR, cls_name)
    if not os.path.isdir(cls_path): continue
    if cls_name.lower() == "valentino":
        print(f"Guardando {cls_name} para testeo de excluidos.")
        for f in os.listdir(cls_path):
            img = cv2.imread(os.path.join(cls_path, f))
            if img is not None: val_valentino_images.append(img)
        continue
    local_classes.append(cls_name)
    cls_idx = len(lfw_names) + len(local_classes) - 1
    for f in os.listdir(cls_path):
        img = cv2.imread(os.path.join(cls_path, f))
        if img is not None:
            X_local_bgr.append(img)
            y_local_raw.append(cls_idx)

all_class_names = lfw_names + local_classes
print(f"Clases Locales: {len(local_classes)} personas. Imágenes: {len(X_local_bgr)}")
print(f"Total de clases finales: {len(all_class_names)}")

Guardando valentino para testeo de excluidos.
Clases Locales: 6 personas. Imágenes: 66
Total de clases finales: 13


### Aplicar el Pipeline (Detección y Alineación) a todo el Dataset
Dado que las fotos son diferentes (algunas de cuerpo entero, otras recortadas), pasaremos todas las imágenes por nuestro pipeline de alineación antes de entrenar.

In [8]:
def process_images_pipeline(images_bgr, labels):
    processed_images = []
    processed_labels = []
    for img, lbl in zip(images_bgr, labels):
        faces = detect_faces(img)
        if len(faces) > 0:
            faces = sorted(faces, key=lambda x: (x[0][2]-x[0][0])*(x[0][3]-x[0][1]), reverse=True)
            box, kps = faces[0]
            aligned = align_face(img, box, kps, face_size=112)
            processed_images.append(cv2.cvtColor(aligned, cv2.COLOR_BGR2RGB))
            processed_labels.append(lbl)
        else:
            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            processed_images.append(cv2.resize(img_rgb, (112, 112)))
            processed_labels.append(lbl)
    return processed_images, processed_labels

print("Procesando imágenes LFW...")
X_lfw_aligned, y_lfw = process_images_pipeline(X_lfw_bgr, y_lfw_raw)

print("Procesando imágenes locales...")
X_local_aligned, y_local = process_images_pipeline(X_local_bgr, y_local_raw)

Procesando imágenes LFW...
Procesando imágenes locales...


### Split de Datos, Undersampling y Oversampling

Dividiremos el conjunto local y el de LFW en Train (80%) y Val (20%).
Luego aplicaremos **Undersampling** a las clases LFW que superen el target (40 imágenes) y **Oversampling** con Data Augmentation a las clases locales que estén por debajo.

De este modo, todas las clases quedan balanceadas en ~40 imágenes de entrenamiento.

In [9]:
# Dividir LFW
X_train_lfw, X_val_lfw, y_train_lfw, y_val_lfw = train_test_split(
    X_lfw_aligned, y_lfw, test_size=0.2, random_state=42, stratify=y_lfw
)

# Dividir Local
X_train_loc, X_val_loc, y_train_loc, y_val_loc = train_test_split(
    X_local_aligned, y_local, test_size=0.2, random_state=42, stratify=y_local
)

TARGET_COUNT = 40

# ── Undersampling de LFW (limitar cada clase a TARGET_COUNT) ──
counter_before_lfw = Counter(y_train_lfw)
X_train_lfw_balanced = []
y_train_lfw_balanced = []

for cls_idx in sorted(counter_before_lfw.keys()):
    indices = [i for i, y in enumerate(y_train_lfw) if y == cls_idx]
    if len(indices) > TARGET_COUNT:
        indices = random.sample(indices, TARGET_COUNT)
    for i in indices:
        X_train_lfw_balanced.append(X_train_lfw[i])
        y_train_lfw_balanced.append(y_train_lfw[i])

print(f"LFW Undersampling: {len(y_train_lfw)} -> {len(y_train_lfw_balanced)} imágenes")

# Contar ANTES del oversampling (locales)
counter_before_local = Counter(y_train_loc)

# ── Oversampling del train local (subir a TARGET_COUNT) ──
X_train_loc_oversampled = list(X_train_loc)
y_train_loc_oversampled = list(y_train_loc)

counter = Counter(y_train_loc)
for cls_idx in set(y_train_loc):
    count = counter[cls_idx]
    if count < TARGET_COUNT:
        needed = TARGET_COUNT - count
        indices = [i for i, y in enumerate(y_train_loc) if y == cls_idx]
        for _ in range(needed):
            idx = random.choice(indices)
            X_train_loc_oversampled.append(X_train_loc[idx])
            y_train_loc_oversampled.append(y_train_loc[idx])

print(f"Local Oversampling: {len(y_train_loc)} -> {len(y_train_loc_oversampled)} imágenes")

# Combinar Final
X_train = X_train_lfw_balanced + X_train_loc_oversampled
y_train = y_train_lfw_balanced + y_train_loc_oversampled

X_val = X_val_lfw + X_val_loc
y_val = y_val_lfw + y_val_loc

print(f"\nTotal Train: {len(X_train)} (Balanced)")
print(f"Total Val: {len(X_val)}")

# Resumen por clase
counter_final = Counter(y_train)
print(f"\nDistribución final del Train:")
for i, name in enumerate(all_class_names):
    print(f"  {name.capitalize():20s}: {counter_final.get(i, 0)} imágenes")


LFW Undersampling: 1030 -> 280 imágenes
Local Oversampling: 52 -> 240 imágenes

Total Train: 520 (Balanced)
Total Val: 272

Distribución final del Train:
  Ariel sharon        : 40 imágenes
  Colin powell        : 40 imágenes
  Donald rumsfeld     : 40 imágenes
  George w bush       : 40 imágenes
  Gerhard schroeder   : 40 imágenes
  Hugo chavez         : 40 imágenes
  Tony blair          : 40 imágenes
  Ambrogi             : 40 imágenes
  Gianfranco          : 40 imágenes
  Gianluca            : 40 imágenes
  Lucas               : 40 imágenes
  Matias              : 40 imágenes
  Roberto             : 40 imágenes


### Visualización del Balanceo de Clases
A continuación visualizamos la distribución **antes** y **después** del balanceo (undersampling de LFW + oversampling de locales), y la distribución final del dataset de entrenamiento.

In [10]:
# ── Gráfico 1: Distribución ANTES del balanceo (todas las clases, train split) ──
counter_before_all = {}
for cls_idx, count in counter_before_lfw.items():
    counter_before_all[all_class_names[cls_idx]] = count
for cls_idx, count in counter_before_local.items():
    counter_before_all[all_class_names[cls_idx]] = count

# Después del balanceo
counter_after_all = {}
counter_final_train = Counter(y_train)
for i, name in enumerate(all_class_names):
    counter_after_all[name] = counter_final_train.get(i, 0)

names_sorted = [n for n in all_class_names if n in counter_before_all]
before_vals = [counter_before_all.get(n, 0) for n in names_sorted]
after_vals = [counter_after_all.get(n, 0) for n in names_sorted]

fig, ax = plt.subplots(figsize=(14, 6))
x = np.arange(len(names_sorted))
w = 0.35
bars1 = ax.bar(x - w/2, before_vals, w, label="Antes (original)", color="#e74c3c", alpha=0.85)
bars2 = ax.bar(x + w/2, after_vals, w, label="Después (balanceado)", color="#2ecc71", alpha=0.85)
ax.set_xlabel("Persona")
ax.set_ylabel("Cantidad de imágenes")
ax.set_title("Balanceo: Undersampling (LFW) + Oversampling (Locales) — Train")
ax.set_xticks(x)
ax.set_xticklabels([n.capitalize() for n in names_sorted], rotation=30, ha="right")
ax.legend()
ax.bar_label(bars1, padding=2, fontsize=8)
ax.bar_label(bars2, padding=2, fontsize=8)
ax.axhline(y=TARGET_COUNT, color="gray", linestyle="--", alpha=0.5, label=f"Target={TARGET_COUNT}")
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
fig.savefig("Graphics/balanceo_antes_despues.png", dpi=150)
plt.show()
plt.close(fig)

# ── Gráfico 2: Distribución final de clases en Train ──
from matplotlib.patches import Patch
fig, ax = plt.subplots(figsize=(12, 5))
colors = ["#3498db" if i < len(lfw_names) else "#e67e22" for i in range(len(all_class_names))]
counts_all = [counter_final_train.get(i, 0) for i in range(len(all_class_names))]
bars = ax.barh(range(len(all_class_names)), counts_all, color=colors, alpha=0.85, edgecolor="white")
ax.set_yticks(range(len(all_class_names)))
ax.set_yticklabels([n.capitalize() for n in all_class_names])
ax.set_xlabel("Cantidad de imágenes (Train)")
ax.set_title("Distribución Final de Clases en el Dataset de Entrenamiento (Balanceado)")
ax.bar_label(bars, padding=3, fontsize=8)
ax.grid(axis="x", alpha=0.3)
ax.legend(handles=[Patch(color="#3498db", label="LFW (undersample)"), Patch(color="#e67e22", label="Local (oversample)")], loc="lower right")
ax.invert_yaxis()
fig.tight_layout()
fig.savefig("Graphics/distribucion_clases_train.png", dpi=150)
plt.show()
plt.close(fig)

print("Gráficos de balanceo guardados en Graphics/")


Gráficos de balanceo guardados en Graphics/


## Data Augmentation y DataLoaders
Dado que duplicamos imágenes exactas en el Oversampling, la **Data Augmentation** agresiva en el DataLoader de entrenamiento es crítica. Cada vez que el modelo vea una de esas imágenes duplicadas, la verá con rotación, brillo, recortes o ruidos distintos.

In [11]:
class InMemoryFaceDataset(Dataset):
    def __init__(self, images_rgb, labels, transform=None):
        self.images = images_rgb
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_np = self.images[idx]
        label = self.labels[idx]
        img_pil = Image.fromarray(img_np)
        if self.transform:
            img_tensor = self.transform(img_pil)
        else:
            img_tensor = transforms.ToTensor()(img_pil)
        return img_tensor, label

train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.1),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0)),
    transforms.RandomGrayscale(p=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_dataset = InMemoryFaceDataset(X_train, y_train, train_transform)
val_dataset = InMemoryFaceDataset(X_val, y_val, val_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

## Entrenamiento del Modelo (Fine-Tuning)

In [12]:
EPOCHS = 50
LEARNING_RATE = 1e-4

num_classes = len(all_class_names)
model = FaceEmbeddingModel(num_classes=num_classes).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)

best_val_acc = 0.0
history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

print(f"Entrenando {EPOCHS} épocas para {num_classes} clases en {DEVICE}...")

for epoch in range(EPOCHS):
    model.train()
    running_loss, correct_train, total_train = 0.0, 0, 0
    for inputs, targets in train_loader:
        inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total_train += targets.size(0)
        correct_train += predicted.eq(targets).sum().item()

    train_acc = 100. * correct_train / total_train
    train_loss = running_loss / len(train_loader)

    # Validación
    model.eval()
    correct_val, total_val, val_loss_sum = 0, 0, 0.0
    with torch.no_grad():
        for inputs, targets in val_loader:
            inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            val_loss_sum += loss.item()
            _, predicted = outputs.max(1)
            total_val += targets.size(0)
            correct_val += predicted.eq(targets).sum().item()

    val_acc = 100. * correct_val / total_val
    val_loss = val_loss_sum / len(val_loader)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)

    print(f"Epoch [{epoch+1}/{EPOCHS}] - Loss: {train_loss:.4f} - Train Acc: {train_acc:.2f}% | Val Loss: {val_loss:.4f} - Val Acc: {val_acc:.2f}%")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        final_model = nn.Sequential(model.backbone, model.embedding_head)
        torch.save(final_model, "models/face_detection.pth")
        print("  -> Modelo guardado en models/face_detection.pth")

print(f"\nMejor Val Accuracy: {best_val_acc:.2f}%")

Entrenando 50 épocas para 13 clases en cuda...
Epoch [1/50] - Loss: 2.5532 - Train Acc: 12.50% | Val Loss: 2.5381 - Val Acc: 12.13%
  -> Modelo guardado en models/face_detection.pth
Epoch [2/50] - Loss: 2.5198 - Train Acc: 20.77% | Val Loss: 2.5180 - Val Acc: 16.18%
  -> Modelo guardado en models/face_detection.pth
Epoch [3/50] - Loss: 2.4773 - Train Acc: 26.92% | Val Loss: 2.4813 - Val Acc: 17.28%
  -> Modelo guardado en models/face_detection.pth
Epoch [4/50] - Loss: 2.3824 - Train Acc: 38.27% | Val Loss: 2.4256 - Val Acc: 18.38%
  -> Modelo guardado en models/face_detection.pth
Epoch [5/50] - Loss: 2.2406 - Train Acc: 45.77% | Val Loss: 2.3229 - Val Acc: 14.71%
Epoch [6/50] - Loss: 2.0364 - Train Acc: 45.58% | Val Loss: 2.1615 - Val Acc: 18.38%
Epoch [7/50] - Loss: 1.8228 - Train Acc: 50.38% | Val Loss: 2.0258 - Val Acc: 23.90%
  -> Modelo guardado en models/face_detection.pth
Epoch [8/50] - Loss: 1.6046 - Train Acc: 55.19% | Val Loss: 1.9412 - Val Acc: 26.84%
  -> Modelo guardado en

## Curvas de Entrenamiento
Visualización de la evolución del Loss y Accuracy durante el entrenamiento, tanto para train como para validación.

In [13]:
# ── Loss vs Epochs ──
fig, ax = plt.subplots(figsize=(10, 5))
epochs_range = range(1, EPOCHS + 1)
ax.plot(epochs_range, history["train_loss"], label="Train Loss", color="#e74c3c", linewidth=2)
ax.plot(epochs_range, history["val_loss"], label="Validation Loss", color="#3498db", linewidth=2)
ax.set_xlabel("Época")
ax.set_ylabel("Loss")
ax.set_title("Loss vs Épocas")
ax.legend(fontsize=12)
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig("Graphics/loss_vs_epochs.png", dpi=150)
plt.show()
plt.close(fig)

# ── Accuracy vs Epochs ──
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(epochs_range, history["train_acc"], label="Train Accuracy", color="#e74c3c", linewidth=2)
ax.plot(epochs_range, history["val_acc"], label="Validation Accuracy", color="#3498db", linewidth=2)
ax.set_xlabel("Época")
ax.set_ylabel("Accuracy (%)")
ax.set_title("Accuracy vs Épocas")
ax.legend(fontsize=12)
ax.grid(alpha=0.3)
ax.set_ylim([0, 105])
fig.tight_layout()
fig.savefig("Graphics/accuracy_vs_epochs.png", dpi=150)
plt.show()
plt.close(fig)

print("Curvas de entrenamiento guardadas en Graphics/")

Curvas de entrenamiento guardadas en Graphics/


## Matriz de Confusión
Evaluamos el modelo sobre el conjunto de validación y generamos la matriz de confusión para visualizar errores de clasificación.

In [14]:
# Obtener predicciones sobre validación
model.eval()
all_preds = []
all_targets = []
all_probs = []

with torch.no_grad():
    for inputs, targets in val_loader:
        inputs = inputs.to(DEVICE)
        outputs = model(inputs)
        probs = torch.softmax(outputs, dim=1)
        _, predicted = outputs.max(1)
        all_preds.extend(predicted.cpu().numpy())
        all_targets.extend(targets.numpy())
        all_probs.extend(probs.cpu().numpy())

all_preds = np.array(all_preds)
all_targets = np.array(all_targets)
all_probs = np.array(all_probs)

# Matriz de confusión
cm = confusion_matrix(all_targets, all_preds)
display_labels = [n.capitalize() for n in all_class_names]

fig, ax = plt.subplots(figsize=(14, 11))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=display_labels)
disp.plot(ax=ax, cmap="Blues", values_format="d", xticks_rotation=45)
ax.set_title("Matriz de Confusión (Conjunto de Validación)", fontsize=14)
fig.tight_layout()
fig.savefig("Graphics/confusion_matrix.png", dpi=150)
plt.show()
plt.close(fig)

print("Matriz de confusión guardada en Graphics/confusion_matrix.png")

Matriz de confusión guardada en Graphics/confusion_matrix.png


## Curva AUC-ROC (Multiclase)
Calculamos la curva ROC para cada clase (One-vs-Rest) y el promedio micro/macro, lo que nos permite evaluar la capacidad discriminativa del modelo.

In [15]:
# Binarizar etiquetas para multiclase
classes_in_val = sorted(list(set(all_targets)))
y_bin = label_binarize(all_targets, classes=classes_in_val)
n_classes_val = y_bin.shape[1]

# Filtrar probabilidades solo para clases presentes en validación
probs_filtered = all_probs[:, classes_in_val]

# Calcular ROC por clase
fpr = {}
tpr = {}
roc_auc = {}

for i in range(n_classes_val):
    fpr[i], tpr[i], _ = roc_curve(y_bin[:, i], probs_filtered[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

# Micro-average
fpr["micro"], tpr["micro"], _ = roc_curve(y_bin.ravel(), probs_filtered.ravel())
roc_auc["micro"] = auc(fpr["micro"], tpr["micro"])

# Graficar
fig, ax = plt.subplots(figsize=(10, 8))
ax.plot(fpr["micro"], tpr["micro"],
        label=f"Micro-average (AUC = {roc_auc['micro']:.4f})",
        color="navy", linestyle="--", linewidth=2.5)

colors = plt.cm.tab20(np.linspace(0, 1, n_classes_val))
for i, color in zip(range(n_classes_val), colors):
    cls_name = all_class_names[classes_in_val[i]].capitalize()
    ax.plot(fpr[i], tpr[i], color=color, linewidth=1.2,
            label=f"{cls_name} (AUC = {roc_auc[i]:.3f})")

ax.plot([0, 1], [0, 1], "k--", alpha=0.3)
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("Curva ROC Multiclase (One-vs-Rest)")
ax.legend(loc="lower right", fontsize=8)
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig("Graphics/auc_roc.png", dpi=150)
plt.show()
plt.close(fig)

print(f"AUC-ROC micro-average: {roc_auc['micro']:.4f}")
print("Curva AUC-ROC guardada en Graphics/auc_roc.png")

AUC-ROC micro-average: 0.9983
Curva AUC-ROC guardada en Graphics/auc_roc.png


## Guardado de Métricas de Entrenamiento
Se exportan las métricas a `train_metrics/` en formato log (legible) y JSON (estructurado).

In [16]:
# ── Guardar JSON ──
metrics_json = {
    "config": {
        "epochs": EPOCHS,
        "batch_size": 32,
        "learning_rate": LEARNING_RATE,
        "embedding_size": 512,
        "face_size": 112,
        "device": DEVICE,
        "num_classes": num_classes,
        "class_names": all_class_names,
        "total_train_images": len(X_train),
        "total_val_images": len(X_val),
    },
    "history": history,
    "best_val_accuracy": best_val_acc,
    "final_train_loss": history["train_loss"][-1],
    "final_val_loss": history["val_loss"][-1],
    "auc_roc_micro": round(roc_auc["micro"], 4),
}

with open("train_metrics/train_metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics_json, f, ensure_ascii=False, indent=2)

# ── Guardar LOG legible ──
best_epoch = int(np.argmax(history["val_acc"])) + 1
timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

with open("train_metrics/train_metrics.log", "w", encoding="utf-8") as f:
    f.write(f"{'='*70}\n")
    f.write(f"  MÉTRICAS DE ENTRENAMIENTO - {timestamp}\n")
    f.write(f"{'='*70}\n\n")
    f.write(f"Dispositivo: {DEVICE}\n")
    f.write(f"Épocas: {EPOCHS}\n")
    f.write(f"Learning Rate: {LEARNING_RATE}\n")
    f.write(f"Batch Size: 32\n")
    f.write(f"Clases: {num_classes}\n")
    f.write(f"Imágenes Train: {len(X_train)}\n")
    f.write(f"Imágenes Val: {len(X_val)}\n")
    f.write(f"Mejor Val Accuracy: {best_val_acc:.2f}% (Época {best_epoch})\n")
    f.write(f"AUC-ROC Micro-Average: {roc_auc['micro']:.4f}\n\n")
    f.write(f"{'Época':>6} {'Train Loss':>12} {'Val Loss':>12} {'Train Acc':>12} {'Val Acc':>12}\n")
    f.write(f"{'-'*56}\n")
    for i in range(EPOCHS):
        f.write(f"{i+1:>6d} {history['train_loss'][i]:>12.4f} {history['val_loss'][i]:>12.4f} {history['train_acc'][i]:>11.2f}% {history['val_acc'][i]:>11.2f}%\n")

print("Métricas guardadas en train_metrics/")
print(f"  - train_metrics.json")
print(f"  - train_metrics.log")

Métricas guardadas en train_metrics/
  - train_metrics.json
  - train_metrics.log


## Pruebas de Generalización: Caso "Valentino" (Desconocido)
Como mencionamos al inicio, no entrenamos con fotos de Valentino para probar si el modelo lo confunde con su hermano Gianluca o si es capaz de detectarlo como un "Desconocido".

Si la similitud supera nuestro umbral (ej. 0.50), significa que el modelo confunde a los hermanos. Si es menor, el modelo aprendió a diferenciarlos verdaderamente.

In [17]:
# Cargar el mejor modelo exportado
eval_model = torch.load("models/face_detection.pth", map_location=DEVICE, weights_only=False)
eval_model.eval()

gianluca_idx = all_class_names.index("gianluca") if "gianluca" in all_class_names else -1

if gianluca_idx != -1 and len(val_valentino_images) > 0:
    gianluca_images = [X_val[i] for i, y in enumerate(y_val) if y == gianluca_idx]
    if len(gianluca_images) > 0:
        emb_gianluca = extract_embedding_from_face(eval_model, gianluca_images[0], val_transform)

        val_img = val_valentino_images[0]
        faces = detect_faces(val_img)
        if faces:
            box, kps = faces[0]
            val_aligned = align_face(val_img, box, kps, face_size=112)
            val_rgb = cv2.cvtColor(val_aligned, cv2.COLOR_BGR2RGB)
            emb_valentino = extract_embedding_from_face(eval_model, val_rgb, val_transform)

            sim_score = similarity(emb_valentino, emb_gianluca)
            print(f"Similitud entre Valentino (Desconocido) y Gianluca (Conocido): {sim_score:.4f}")
            if sim_score < 0.5:
                print("¡Éxito! El modelo los diferencia correctamente y tratará a Valentino como desconocido.")
            else:
                print("Alerta: El modelo los considera la misma persona (Sobreajuste genético/Falso Positivo).")
else:
    print("No se encontraron imágenes de Gianluca o Valentino para la prueba extrema.")

Similitud entre Valentino (Desconocido) y Gianluca (Conocido): 0.9482
Alerta: El modelo los considera la misma persona (Sobreajuste genético/Falso Positivo).


## Evaluación de Inferencia Externa

En esta sección evaluamos el modelo entrenado sobre imágenes **completamente externas** almacenadas en la carpeta `Inferencia/`.

Las subcarpetas corresponden a:
- **Personas conocidas** (parte del dataset de entrenamiento): ambrogi, gianfranco, gianluca, lucas, matias, roberto
- **Personas desconocidas**: valentino (excluido del entrenamiento), otros
- **Imágenes con múltiples personas**: varias_personas

El umbral de similitud para considerar a alguien como "conocido" es **0.75**.

In [18]:
# ── Configuración de Inferencia ──
INFERENCIA_DIR = "Inferencia"
SIMILARITY_THRESHOLD = 0.75

# Personas que están en el dataset de entrenamiento
KNOWN_PERSONS = ["ambrogi", "gianfranco", "gianluca", "lucas", "matias", "roberto"]
UNKNOWN_PERSONS = ["valentino", "otros"]
MULTI_FACE = ["varias_personas"]

# Cargar el mejor modelo exportado
eval_model = torch.load("models/face_detection.pth", map_location=DEVICE, weights_only=False)
eval_model.eval()

# Construir banco de embeddings de referencia (promedio por clase del val set)
ref_embeddings = {}
for cls_idx, cls_name in enumerate(all_class_names):
    cls_images = [X_val[i] for i, y in enumerate(y_val) if y == cls_idx]
    if not cls_images:
        cls_images = [X_train[i] for i, y in enumerate(y_train) if y == cls_idx]
    if cls_images:
        embs = []
        for img in cls_images[:10]:  # max 10 para referencia
            emb = extract_embedding_from_face(eval_model, img, val_transform)
            embs.append(emb)
        ref_embeddings[cls_name] = np.mean(embs, axis=0)

print(f"Banco de referencia: {len(ref_embeddings)} clases")
print(f"Umbral de similitud: {SIMILARITY_THRESHOLD}")

Banco de referencia: 13 clases
Umbral de similitud: 0.75


In [19]:
def identify_face(face_rgb, ref_embeddings, threshold=0.75):
    """Identifica una cara contra el banco de referencia.
    Retorna (nombre, similitud) o ('desconocido', mejor_sim) si no supera el umbral."""
    emb = extract_embedding_from_face(eval_model, face_rgb, val_transform)
    best_name = "desconocido"
    best_sim = -1
    for name, ref_emb in ref_embeddings.items():
        sim = similarity(emb, ref_emb)
        if sim > best_sim:
            best_sim = sim
            best_name = name
    if best_sim < threshold:
        return "desconocido", best_sim
    return best_name, best_sim


def process_inference_folder(folder_path):
    """Procesa todas las imágenes de una carpeta de inferencia.
    Retorna lista de dict con resultados por imagen."""
    results = []
    if not os.path.isdir(folder_path):
        return results
    for fname in sorted(os.listdir(folder_path)):
        fpath = os.path.join(folder_path, fname)
        img_bgr = cv2.imread(fpath)
        if img_bgr is None:
            continue
        faces = detect_faces(img_bgr)
        if not faces:
            results.append({"file": fname, "faces": [], "note": "No se detectaron caras"})
            continue
        face_results = []
        for box, kps in faces:
            aligned = align_face(img_bgr, box, kps, face_size=112)
            face_rgb = cv2.cvtColor(aligned, cv2.COLOR_BGR2RGB)
            pred_name, pred_sim = identify_face(face_rgb, ref_embeddings, SIMILARITY_THRESHOLD)
            face_results.append({"predicted": pred_name, "similarity": round(pred_sim, 4)})
        results.append({"file": fname, "faces": face_results})
    return results

print("Funciones de inferencia definidas.")

Funciones de inferencia definidas.


### Evaluación sobre Personas Conocidas y Desconocidas

In [20]:
# ── Procesar todas las carpetas de inferencia ──
all_inference_results = {}
all_folders = KNOWN_PERSONS + UNKNOWN_PERSONS + MULTI_FACE

for folder_name in all_folders:
    folder_path = os.path.join(INFERENCIA_DIR, folder_name)
    if not os.path.isdir(folder_path):
        print(f"  [SKIP] {folder_name}/ no encontrada")
        continue
    results = process_inference_folder(folder_path)
    all_inference_results[folder_name] = results
    n_images = len(results)
    n_faces = sum(len(r["faces"]) for r in results)
    print(f"  {folder_name}: {n_images} imágenes, {n_faces} caras detectadas")

print(f"\nTotal carpetas procesadas: {len(all_inference_results)}")

  ambrogi: 15 imágenes, 16 caras detectadas
  gianfranco: 23 imágenes, 22 caras detectadas
  gianluca: 18 imágenes, 20 caras detectadas
  lucas: 15 imágenes, 18 caras detectadas
  matias: 20 imágenes, 21 caras detectadas
  roberto: 15 imágenes, 17 caras detectadas
  valentino: 2 imágenes, 2 caras detectadas
  otros: 9 imágenes, 9 caras detectadas
  varias_personas: 26 imágenes, 57 caras detectadas

Total carpetas procesadas: 9


In [21]:
# ── Métricas para personas CONOCIDAS ──
print("="*60)
print("  RESULTADOS: PERSONAS CONOCIDAS")
print("="*60)

known_correct = 0
known_total = 0
known_details = {}

for person in KNOWN_PERSONS:
    if person not in all_inference_results:
        continue
    results = all_inference_results[person]
    correct = 0
    total = 0
    for r in results:
        for face in r["faces"]:
            total += 1
            if face["predicted"] == person:
                correct += 1
    acc = (correct / total * 100) if total > 0 else 0
    known_details[person] = {"correct": correct, "total": total, "accuracy": round(acc, 1)}
    known_correct += correct
    known_total += total
    print(f"  {person.capitalize():15s}: {correct}/{total} correctas ({acc:.1f}%)")

overall_known_acc = (known_correct / known_total * 100) if known_total > 0 else 0
print(f"\n  TOTAL Conocidos: {known_correct}/{known_total} ({overall_known_acc:.1f}%)")

  RESULTADOS: PERSONAS CONOCIDAS
  Ambrogi        : 11/16 correctas (68.8%)
  Gianfranco     : 19/22 correctas (86.4%)
  Gianluca       : 14/20 correctas (70.0%)
  Lucas          : 14/18 correctas (77.8%)
  Matias         : 17/21 correctas (81.0%)
  Roberto        : 13/17 correctas (76.5%)

  TOTAL Conocidos: 88/114 (77.2%)


In [22]:
# ── Métricas para personas DESCONOCIDAS ──
print("="*60)
print("  RESULTADOS: PERSONAS DESCONOCIDAS")
print("="*60)

unknown_correct = 0  # correctamente identificado como desconocido
unknown_total = 0
unknown_details = {}

for person in UNKNOWN_PERSONS:
    if person not in all_inference_results:
        continue
    results = all_inference_results[person]
    correct = 0
    total = 0
    false_positives = []  # con quién los confunde
    for r in results:
        for face in r["faces"]:
            total += 1
            if face["predicted"] == "desconocido":
                correct += 1
            else:
                false_positives.append(face["predicted"])
    acc = (correct / total * 100) if total > 0 else 0
    unknown_details[person] = {
        "correct_unknown": correct, "total": total,
        "rejection_rate": round(acc, 1), "false_positives": Counter(false_positives)
    }
    unknown_correct += correct
    unknown_total += total
    print(f"  {person.capitalize():15s}: {correct}/{total} rechazados correctamente ({acc:.1f}%)")
    if false_positives:
        fp_summary = ", ".join(f"{n}({c})" for n, c in Counter(false_positives).most_common())
        print(f"    Confundido con: {fp_summary}")

overall_rejection = (unknown_correct / unknown_total * 100) if unknown_total > 0 else 0
print(f"\n  TOTAL Desconocidos rechazados: {unknown_correct}/{unknown_total} ({overall_rejection:.1f}%)")

  RESULTADOS: PERSONAS DESCONOCIDAS
  Valentino      : 0/2 rechazados correctamente (0.0%)
    Confundido con: gianluca(2)
  Otros          : 0/9 rechazados correctamente (0.0%)
    Confundido con: gianluca(5), matias(3), roberto(1)

  TOTAL Desconocidos rechazados: 0/11 (0.0%)


In [23]:
# ── Resultados para VARIAS PERSONAS ──
print("="*60)
print("  RESULTADOS: IMÁGENES CON VARIAS PERSONAS")
print("="*60)

if "varias_personas" in all_inference_results:
    for r in all_inference_results["varias_personas"]:
        print(f"\n  📷 {r['file']}:")
        if not r["faces"]:
            print(f"    No se detectaron caras.")
            if "note" in r:
                print(f"    Nota: {r['note']}")
        else:
            print(f"    Caras detectadas: {len(r['faces'])}")
            for i, face in enumerate(r["faces"]):
                status = "✅ Conocido" if face["predicted"] != "desconocido" else "❌ Desconocido"
                print(f"    Cara {i+1}: {face['predicted'].capitalize()} (sim={face['similarity']}) {status}")
else:
    print("  Carpeta varias_personas/ no encontrada.")

  RESULTADOS: IMÁGENES CON VARIAS PERSONAS

  📷 varias_personas.jpg:
    Caras detectadas: 2
    Cara 1: Ambrogi (sim=0.939) ✅ Conocido
    Cara 2: Gianfranco (sim=0.9743) ✅ Conocido

  📷 varias_personas_02.jpg:
    Caras detectadas: 2
    Cara 1: Gianfranco (sim=0.9848) ✅ Conocido
    Cara 2: Gianfranco (sim=0.8073) ✅ Conocido

  📷 varias_personas_03.jpg:
    Caras detectadas: 2
    Cara 1: Lucas (sim=0.9649) ✅ Conocido
    Cara 2: Gianluca (sim=0.8499) ✅ Conocido

  📷 varias_personas_04.jpg:
    Caras detectadas: 3
    Cara 1: Lucas (sim=0.976) ✅ Conocido
    Cara 2: Gianluca (sim=0.9535) ✅ Conocido
    Cara 3: Lucas (sim=0.9619) ✅ Conocido

  📷 varias_personas_05.jpg:
    Caras detectadas: 2
    Cara 1: Gianluca (sim=0.8917) ✅ Conocido
    Cara 2: Gianluca (sim=0.8916) ✅ Conocido

  📷 varias_personas_06.jpg:
    Caras detectadas: 2
    Cara 1: Gianluca (sim=0.9685) ✅ Conocido
    Cara 2: Gianluca (sim=0.9415) ✅ Conocido

  📷 varias_personas_07.jpeg:
    Caras detectadas: 3
    Cara 

### Gráficos de Inferencia

In [24]:
# ── Gráfico: Distribución de similitudes por categoría ──
known_sims = []
unknown_sims = []

for person in KNOWN_PERSONS:
    if person not in all_inference_results: continue
    for r in all_inference_results[person]:
        for face in r["faces"]:
            known_sims.append(face["similarity"])

for person in UNKNOWN_PERSONS:
    if person not in all_inference_results: continue
    for r in all_inference_results[person]:
        for face in r["faces"]:
            unknown_sims.append(face["similarity"])

fig, ax = plt.subplots(figsize=(10, 5))
if known_sims:
    ax.hist(known_sims, bins=20, alpha=0.7, label=f"Conocidos (n={len(known_sims)})", color="#2ecc71", edgecolor="white")
if unknown_sims:
    ax.hist(unknown_sims, bins=20, alpha=0.7, label=f"Desconocidos (n={len(unknown_sims)})", color="#e74c3c", edgecolor="white")
ax.axvline(x=SIMILARITY_THRESHOLD, color="black", linestyle="--", linewidth=2, label=f"Umbral={SIMILARITY_THRESHOLD}")
ax.set_xlabel("Similitud Coseno (máxima vs referencia)")
ax.set_ylabel("Frecuencia")
ax.set_title("Distribución de Similitudes: Conocidos vs Desconocidos")
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig("Graphics/inferencia_similitudes.png", dpi=150)
plt.show()
plt.close(fig)
print("Gráfico de similitudes guardado en Graphics/inferencia_similitudes.png")

Gráfico de similitudes guardado en Graphics/inferencia_similitudes.png


In [25]:
# ── Gráfico: Accuracy por persona (barras) ──
persons_plot = []
accs_plot = []
colors_plot = []

for person in KNOWN_PERSONS:
    if person in known_details:
        persons_plot.append(person.capitalize())
        accs_plot.append(known_details[person]["accuracy"])
        colors_plot.append("#2ecc71")

for person in UNKNOWN_PERSONS:
    if person in unknown_details:
        persons_plot.append(person.capitalize() + "\n(desconocido)")
        accs_plot.append(unknown_details[person]["rejection_rate"])
        colors_plot.append("#e74c3c")

if persons_plot:
    fig, ax = plt.subplots(figsize=(12, 5))
    bars = ax.bar(range(len(persons_plot)), accs_plot, color=colors_plot, alpha=0.85, edgecolor="white")
    ax.set_xticks(range(len(persons_plot)))
    ax.set_xticklabels(persons_plot, rotation=30, ha="right")
    ax.set_ylabel("Tasa de acierto (%)")
    ax.set_title("Evaluación de Inferencia Externa: Accuracy por Persona")
    ax.set_ylim([0, 105])
    ax.bar_label(bars, fmt="%.1f%%", padding=3, fontsize=9)
    ax.grid(axis="y", alpha=0.3)
    from matplotlib.patches import Patch
    ax.legend(handles=[
        Patch(color="#2ecc71", label="Conocido (accuracy)"),
        Patch(color="#e74c3c", label="Desconocido (rechazo correcto)")
    ])
    fig.tight_layout()
    fig.savefig("Graphics/inferencia_accuracy.png", dpi=150)
    plt.show()
    plt.close(fig)
    print("Gráfico de accuracy guardado en Graphics/inferencia_accuracy.png")
else:
    print("No hay datos suficientes para graficar.")

Gráfico de accuracy guardado en Graphics/inferencia_accuracy.png


In [26]:
# ── Guardar reporte JSON de inferencia ──
inference_report = {
    "threshold": SIMILARITY_THRESHOLD,
    "known_persons": {
        "overall_accuracy": round(overall_known_acc, 2),
        "details": known_details
    },
    "unknown_persons": {
        "overall_rejection_rate": round(overall_rejection, 2),
        "details": {k: {kk: vv for kk, vv in v.items() if kk != 'false_positives'} for k, v in unknown_details.items()}
    },
    "varias_personas": all_inference_results.get("varias_personas", []),
}

with open("train_metrics/inference_report.json", "w", encoding="utf-8") as f:
    json.dump(inference_report, f, ensure_ascii=False, indent=2)

print("Reporte de inferencia guardado en train_metrics/inference_report.json")
print(f"\n{'='*60}")
print(f"  RESUMEN FINAL")
print(f"{'='*60}")
print(f"  Conocidos  - Accuracy:        {overall_known_acc:.1f}%")
print(f"  Desconocidos - Tasa rechazo:   {overall_rejection:.1f}%")
print(f"  Umbral de similitud:           {SIMILARITY_THRESHOLD}")
print(f"{'='*60}")

Reporte de inferencia guardado en train_metrics/inference_report.json

  RESUMEN FINAL
  Conocidos  - Accuracy:        77.2%
  Desconocidos - Tasa rechazo:   0.0%
  Umbral de similitud:           0.75
